In [17]:
import chess.pgn
import pandas as pd
import io


In [15]:
df = pd.read_csv("games.csv")

In [16]:
df

,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
0,TZJHLljE,False,1.504210e+12,1.504210e+12,13,outoftime,white,15+2,bourgris,1500,a-00,1191,d4 d5 c4 c6 cxd5 e6 dxe6 fxe6 Nf3 Bb4+ Nc3 Ba5...,D10,Slav Defense: Exchange Variation,5
1,l1NXvwaE,True,1.504130e+12,1.504130e+12,16,resign,black,5+10,a-00,1322,skinnerua,1261,d4 Nc6 e4 e5 f4 f6 dxe5 fxe5 fxe5 Nxe5 Qd4 Nc6...,B00,Nimzowitsch Defense: Kennedy Variation,4
2,mIICvQHh,True,1.504130e+12,1.504130e+12,61,mate,white,5+10,ischia,1496,a-00,1500,e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...,C20,King's Pawn Game: Leonardis Variation,3
3,kWKvrqYL,True,1.504110e+12,1.504110e+12,61,mate,white,20+0,daniamurashov,1439,adivanov2009,1454,d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...,D02,Queen's Pawn Game: Zukertort Variation,3
4,9tXo1AUZ,True,1.504030e+12,1.504030e+12,95,mate,white,30+3,nik221107,1523,adivanov2009,1469,e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...,C41,Philidor Defense,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20053,EfqH7VVH,True,1.499791e+12,1.499791e+12,24,resign,white,10+10,belcolt,1691,jamboger,1220,d4 f5 e3 e6 Nf3 Nf6 Nc3 b6 Be2 Bb7 O-O Be7 Ne5...,A80,Dutch Defense,2
20054,WSJDhbPl,True,1.499698e+12,1.499699e+12,82,mate,black,10+0,jamboger,1233,farrukhasomiddinov,1196,d4 d6 Bf4 e5 Bg3 Nf6 e3 exd4 exd4 d5 c3 Bd6 Bd...,A41,Queen's Pawn,2
20055,yrAas0Kj,True,1.499698e+12,1.499698e+12,35,mate,white,10+0,jamboger,1219,schaaksmurf3,1286,d4 d5 Bf4 Nc6 e3 Nf6 c3 e6 Nf3 Be7 Bd3 O-O Nbd...,D00,Queen's Pawn Game: Mason Attack,3
20056,b0v4tRyF,True,1.499696e+12,1.499697e+12,109,resign,white,10+0,marcodisogno,1360,jamboger,1227,e4 d6 d4 Nf6 e5 dxe5 dxe5 Qxd1+ Kxd1 Nd5 c4 Nb...,B07,Pirc Defense,4


In [20]:
def parse_pgn_from_string(pgn_string) -> chess.pgn.Game:
    pgn_io = io.StringIO(pgn_string)
    game = chess.pgn.read_game(pgn_io)
    return game

def result_to_label(result: str) -> float:
    if result == "1-0":
        return 1.0
    elif result == "0-1":
        return -1.0
    else:
        return 0.0
    
def generate_training_examples_from_game(game: chess.pgn.Game) -> list:
    """
    Goes through the game and generates training examples
    each example is a tuple of (board, move, label)
        Board: the board before the move
        Move: the move made
        Label: the result of the game
    """
    examples = []
    result = game.headers.get("Result", "1/2-1/2")
    label = result_to_label(result)

    board = game.board()
    for move in game.mainline_moves():
        board_state = board.fen()
        examples.append((board_state, move.uci(), label))
        board.push(move)
    return examples

def get_training_data_from_csv(csv_oath: str) -> list:
    df = pd.read_csv(csv_oath)
    all_examples = []
    for index, row in df.iterrows():
        game = parse_pgn_from_string(row["moves"])
        if game is not None:
            examples = generate_training_examples_from_game(game)
            all_examples.extend(examples)
    return all_examples


In [21]:
training_data = get_training_data_from_csv("games.csv")
print(len(training_data))

1212827


In [25]:
training_data[0][2]

0.0

In [26]:
# Convert training_data to a DataFrame
training_df = pd.DataFrame(training_data, columns=['Board', 'Move', 'Label'])

# Save the DataFrame to a CSV file
training_df.to_csv('training_data.csv', index=False)

In [27]:
training_data

[('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'd2d4', 0.0),
 ('rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq - 0 1', 'd7d5', 0.0),
 ('rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBNR w KQkq - 0 2', 'c2c4', 0.0),
 ('rnbqkbnr/ppp1pppp/8/3p4/2PP4/8/PP2PPPP/RNBQKBNR b KQkq - 0 2', 'c7c6', 0.0),
 ('rnbqkbnr/pp2pppp/2p5/3p4/2PP4/8/PP2PPPP/RNBQKBNR w KQkq - 0 3',
  'c4d5',
  0.0),
 ('rnbqkbnr/pp2pppp/2p5/3P4/3P4/8/PP2PPPP/RNBQKBNR b KQkq - 0 3', 'e7e6', 0.0),
 ('rnbqkbnr/pp3ppp/2p1p3/3P4/3P4/8/PP2PPPP/RNBQKBNR w KQkq - 0 4',
  'd5e6',
  0.0),
 ('rnbqkbnr/pp3ppp/2p1P3/8/3P4/8/PP2PPPP/RNBQKBNR b KQkq - 0 4', 'f7e6', 0.0),
 ('rnbqkbnr/pp4pp/2p1p3/8/3P4/8/PP2PPPP/RNBQKBNR w KQkq - 0 5', 'g1f3', 0.0),
 ('rnbqkbnr/pp4pp/2p1p3/8/3P4/5N2/PP2PPPP/RNBQKB1R b KQkq - 1 5', 'f8b4', 0.0),
 ('rnbqk1nr/pp4pp/2p1p3/8/1b1P4/5N2/PP2PPPP/RNBQKB1R w KQkq - 2 6',
  'b1c3',
  0.0),
 ('rnbqk1nr/pp4pp/2p1p3/8/1b1P4/2N2N2/PP2PPPP/R1BQKB1R b KQkq - 3 6',
  'b4a5',
  0.0),
 ('rnbqk1nr/pp4pp/2p1p3/b